## Task #1 - News Topic Classifier Using BERT    

### `Objective`:    
Fine-tune a transformer model (e.g., BERT) to classify news headlines into topic categories.    

`Dataset`:    
AG News Dataset (Available on Hugging Face Datasets)  

`Instructions`:    
- Tokenize and preprocess the dataset    
- Fine-tune the bert-base-uncased model using Hugging Face Transformers   
- Evaluate the model using accuracy and F1-score    
- Deploy the model using Streamlit or Gradio for live interaction    

---------------

## Problem Statement
- Manually categorizing news articles by topic is slow and doesn't scale for platforms handling large volumes of content daily, creating a need for automated topic classification.

## Goal
- Fine-tune bert-base-uncased on the AG News dataset to classify news text into four categories (World, Sports, Business, Sci/Tech), then deploy it as an interactive app for real-time predictions.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "ZainAliKhanZAK/bert_agnews_classifier"          # fine-tuned model weights live here
TOKENIZER_SOURCE = "ZainAliKhanZAK/bert_agnews_classifier"      # always load tokenizer from the original checkpoint
LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]

def load():
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_SOURCE)  # fixed: not from MODEL_PATH
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    model.eval()
    if torch.cuda.is_available():
        model.to("cuda")
    return tokenizer, model

def predict(text, tokenizer, model):
    device = model.device
    inputs = tokenizer(text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]
    predicted_idx = torch.argmax(probs).item()
    return LABEL_NAMES[predicted_idx], probs[predicted_idx].item(), probs

def main():
    tokenizer, model = load()

    # Sanity check — confirms tokenizer isn't producing all-[UNK] tokens
    check = tokenizer("Lakers defeat Celtics in overtime thriller")["input_ids"]
    unk_count = check[1:-1].count(tokenizer.unk_token_id)
    print(f"Sanity check tokens: {check}")
    assert unk_count < 3, "Tokenizer still producing mostly [UNK] tokens — something's still wrong."
    print("Tokenizer sanity check passed.\n")

    test_headlines = [
        "Apple unveils new AI chip for next-gen iPhones",
        "Manchester United wins the Premier League title",
        "Stock markets rally after central bank rate cut",
        "United Nations calls for ceasefire amid rising tensions",
    ]

    for text in test_headlines:
        label, confidence, probs = predict(text, tokenizer, model)
        print(f"Text: {text}")
        print(f"  -> Predicted: {label} ({confidence*100:.1f}%)")
        print(f"  -> All scores: " + ", ".join(f"{LABEL_NAMES[i]}={probs[i]:.3f}" for i in range(4)))
        print()

    print("Type a headline to test manually (or 'quit' to exit):")
    while True:
        user_text = input("> ")
        if user_text.strip().lower() == "quit":
            break
        label, confidence, probs = predict(user_text, tokenizer, model)
        print(f"  -> {label} ({confidence*100:.1f}%)\n")

if __name__ == "__main__":
    main()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3567.61it/s]


Sanity check tokens: [101, 18264, 4154, 23279, 1999, 12253, 10874, 102]
Tokenizer sanity check passed.

Text: Apple unveils new AI chip for next-gen iPhones
  -> Predicted: Sci/Tech (99.6%)
  -> All scores: World=0.001, Sports=0.000, Business=0.003, Sci/Tech=0.996

Text: Manchester United wins the Premier League title
  -> Predicted: World (93.8%)
  -> All scores: World=0.938, Sports=0.060, Business=0.002, Sci/Tech=0.000

Text: Stock markets rally after central bank rate cut
  -> Predicted: Business (99.1%)
  -> All scores: World=0.009, Sports=0.000, Business=0.991, Sci/Tech=0.000

Text: United Nations calls for ceasefire amid rising tensions
  -> Predicted: World (99.9%)
  -> All scores: World=0.999, Sports=0.000, Business=0.000, Sci/Tech=0.000

Type a headline to test manually (or 'quit' to exit):
  -> Business (39.6%)

  -> Sci/Tech (99.6%)

  -> World (91.5%)

  -> World (72.2%)

  -> Business (99.9%)



# Final Results
- Fine-tuned BERT for 3 epochs, achieving 94.6% accuracy and 94.6% macro F1-score on the test set
- Verified predictions through CLI testing on both dataset samples and custom headlines
- Deployed as a Streamlit app, with the model hosted on Hugging Face Hub and code pushed to GitHub for public deployment